In [2]:
from pathlib import Path
import sys
import os
import pandas as pd

# ===== 1) 项目根目录：按你的实际位置改 =====
project_root = Path(r"D:\Study Abroad\course\DSCI498\Project")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ===== 2) 导入你队友已经写好的函数 =====
from src.preprocessing.unpack_osz import unpack_osz_files
from src.preprocessing.osutaiko_parser import parse_osu_file_to_jsons

# ===== 3) 路径配置 =====
raw_glob = str(project_root / "data" / "raw" / "*.osz")
unpacked_root = project_root / "data" / "unpacked"

unpacked_root.mkdir(parents=True, exist_ok=True)

# ===== 4) 先解包所有 .osz =====
extracted = unpack_osz_files(
    source_glob=raw_glob,
    destination_root=str(unpacked_root),
    overwrite=False,
)

print(f"Unpacked {len(extracted)} archive(s).")

# ===== 5) 工具函数：读取 .osu 的 mode =====
def read_mode_from_osu(osu_path: Path):
    try:
        text = osu_path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        return None

    for line in text.splitlines():
        line = line.strip()
        if line.startswith("Mode:"):
            return line.split(":", 1)[1].strip()
    return None

# ===== 6) 遍历所有解包后的 .osu，只处理 Mode: 1 =====
rows = []

map_dirs = [p for p in unpacked_root.iterdir() if p.is_dir()]
print(f"Found {len(map_dirs)} unpacked map folder(s).")

for map_dir in map_dirs:
    map_id = map_dir.name
    parsed_dir = map_dir / "parsed"
    parsed_dir.mkdir(exist_ok=True)

    osu_files = list(map_dir.glob("*.osu"))

    for osu_path in osu_files:
        mode = read_mode_from_osu(osu_path)

        # 只处理 taiko
        if mode != "1":
            continue

        map_name = osu_path.stem

        try:
            parse_osu_file_to_jsons(
                osu_path=osu_path,
                out_dir=parsed_dir,
                include_bpm_events=True,
            )

            notes_path = parsed_dir / f"{map_name}.notes.json"
            timing_path = parsed_dir / f"{map_name}.timing.json"
            metadata_path = parsed_dir / f"{map_name}.metadata.json"

            rows.append({
                "map_id": map_id,
                "map_name": map_name,
                "osu_file": osu_path.name,
                "mode": mode,
                "notes_json": str(notes_path),
                "timing_json": str(timing_path),
                "metadata_json": str(metadata_path),
                "parsed_ok": notes_path.exists() and timing_path.exists() and metadata_path.exists(),
            })

            print(f"[OK] Parsed: {map_id} / {osu_path.name}")

        except Exception as e:
            rows.append({
                "map_id": map_id,
                "map_name": osu_path.stem,
                "osu_file": osu_path.name,
                "mode": mode,
                "notes_json": None,
                "timing_json": None,
                "metadata_json": None,
                "parsed_ok": False,
                "error": str(e),
            })
            print(f"[FAIL] {map_id} / {osu_path.name} -> {e}")

# ===== 7) 保存索引表 =====
df = pd.DataFrame(rows)
index_path = project_root / "data" / "parsed_index.csv"
df.to_csv(index_path, index=False, encoding="utf-8-sig")

print("\nDone.")
print(f"Total Taiko maps processed: {len(df)}")
print(f"Success count: {df['parsed_ok'].sum()}")
print(f"Index saved to: {index_path}")

print("\nPreview:")
print(df.head(10).to_string())

Unpacking .osz files: 100%|████████████████████████████████████████████████████████| 100/100 [00:02<00:00, 35.27file/s]


Unpacked 100 archive(s).
Found 100 unpacked map folder(s).
[OK] Parsed: 1024653 / Mizukara o Enshutsu Suru Otome no Kai - Girlish Lover (yuki_momoiro722) [Genjuro's Futsuu].osu
[OK] Parsed: 1024653 / Mizukara o Enshutsu Suru Otome no Kai - Girlish Lover (yuki_momoiro722) [Muzukashii].osu
[OK] Parsed: 1024653 / Mizukara o Enshutsu Suru Otome no Kai - Girlish Lover (yuki_momoiro722) [Oni].osu
[OK] Parsed: 1057540 / UNDEAD CORPORATION - Under the scarlet sky pt,2 inst (Melissa) [Futsuu].osu
[OK] Parsed: 1057540 / UNDEAD CORPORATION - Under the scarlet sky pt,2 inst (Melissa) [Scarlet Devil].osu
[OK] Parsed: 1086408 / Himeringo - Yonjuunana (Zhuosh) [Futsuu].osu
[OK] Parsed: 1086408 / Himeringo - Yonjuunana (Zhuosh) [Oni].osu
[OK] Parsed: 1096486 / Corpo-Mente & Igorrr - Arsalein  Pavor Nocturnus [2015]  Equus (3san) [Nightmare].osu
[OK] Parsed: 1103294 / Yatoki Tsukasa feat. Misawa Aki - Starlight Vision (4sbet1) [Inner Oni].osu
[OK] Parsed: 1112418 / Jackal Queenston - Rubber Band (Midna

[OK] Parsed: 173373 / ZAQ - VOICE (TV Size) (Akemi_Homura) [Oni].osu
[OK] Parsed: 1836870 / Silentroom vs. Frums - Aegleseeker (Afterworld Full Version) (T w i g) [Beyond Space (Light of Another World)].osu
[OK] Parsed: 1841040 / Jinjin - pi (Nytrocide_) [circle].osu
[OK] Parsed: 1913307 / Grabbitz - Die For You (Burak) [AhmetUrmub2008 & DJ Gyrotta Zao's Instalock Jett Kantan].osu
[OK] Parsed: 1913307 / Grabbitz - Die For You (Burak) [Drop Oni Phantom Oni].osu
[OK] Parsed: 1913307 / Grabbitz - Die For You (Burak) [Heal Me Please Muzukashii].osu
[OK] Parsed: 1913307 / Grabbitz - Die For You (Burak) [melatoninSPAMMER's KAYO Abuse Futsuu].osu
[OK] Parsed: 1966609 / Shizuru (CV Nabatame Hitomi), Rino (CV Asumi Kana) - SUPER CHOCOLATE (Game Ver.) ([-E S I A-]) [4SBET1's FUTSUU].osu
[OK] Parsed: 1966609 / Shizuru (CV Nabatame Hitomi), Rino (CV Asumi Kana) - SUPER CHOCOLATE (Game Ver.) ([-E S I A-]) [MUZUKASHII].osu
[OK] Parsed: 1966609 / Shizuru (CV Nabatame Hitomi), Rino (CV Asumi Kana) - S

[OK] Parsed: 2336715 / Can Bardd - Une Couronne de Branches (Eltigant) [Aboutissement d'une Vie de Desarroi].osu
[OK] Parsed: 2340250 / Camellia - I Can Fly In The Universe (Genjuro) [Oni].osu
[OK] Parsed: 2358745 / DIVELA feat. Kagamine Rin - Dawn Alone With You (SN707) [Daybreak Oni].osu
[OK] Parsed: 2358745 / DIVELA feat. Kagamine Rin - Dawn Alone With You (SN707) [Futsuu].osu
[OK] Parsed: 2358745 / DIVELA feat. Kagamine Rin - Dawn Alone With You (SN707) [Muzukashii].osu
[OK] Parsed: 2358745 / DIVELA feat. Kagamine Rin - Dawn Alone With You (SN707) [Oni].osu
[OK] Parsed: 2374632 / Yoko Shimomura - Teehee Valley (Ness_Okey) [Futsuu].osu
[OK] Parsed: 2374632 / Yoko Shimomura - Teehee Valley (Ness_Okey) [Kantan].osu
[OK] Parsed: 2374632 / Yoko Shimomura - Teehee Valley (Ness_Okey) [Muzukashii].osu
[OK] Parsed: 2374632 / Yoko Shimomura - Teehee Valley (Ness_Okey) [Oni].osu
[OK] Parsed: 2383706 / STEREO DIVE FOUNDATION - Genesis ([-E S I A-]) [Futsuu].osu
[OK] Parsed: 2383706 / STEREO DI

[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Futsuu].osu
[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Inner Oni].osu
[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Kantan].osu
[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Madoka & Homura].osu
[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Muzukashii].osu
[OK] Parsed: 782253 / Helblinde - Grief & Malice (Rhytoly) [Oni].osu
[OK] Parsed: 834231 / odaxelagnia - #sawg (Wan Bushi Remix) (Kqrth) [#oni].osu
[OK] Parsed: 881951 / ColorsSlash - Colors Power ni Omakasero! (Angeart remix) (Shallty) [ColorsOni].osu
[OK] Parsed: 914056 / Petit Rabbit's - Happiness Encore (Tyistiana) [Blacken's Oni].osu
[OK] Parsed: 914056 / Petit Rabbit's - Happiness Encore (Tyistiana) [Fapu's Futsuu].osu
[OK] Parsed: 914056 / Petit Rabbit's - Happiness Encore (Tyistiana) [Fapu's Muzukashii].osu
[OK] Parsed: 914056 / Petit Rabbit's - Happiness Encore (Tyistiana) [Futsuu].osu
[OK] Parsed: 914056 / 